# Module 16: Web Dashboard

## Mission Briefing

Last module, your Pico connected to WiFi and got an address on the network. But it was just sitting there quietly, like a house with the lights off.

Today, you turn the lights on. Your Pico is about to become a **web server** -- a tiny website that you can open on your phone, your laptop, or any device on the same WiFi.

And it won't just show information. It will have **buttons** that control a real LED.

```
   +----------+                    +----------+
   |  Phone   |   "Turn LED on!"   | Pico 2W  |
   | (browser)|  ===============>  | (server) |
   |          |                    |          |
   |  [ ON ]  |  <===============  |   LED    |
   | [ OFF ]  |   "Here's a page   |   (*) ON |
   |          |    LED is ON"      |          |
   +----------+                    +----------+
       WiFi         HTTP              GPIO
```

No app to install. No special software. Just open a web browser, type the Pico's IP address, and you're in control.

---
## Science Spotlight: How the Web Works

What if you could control your LED from your phone -- without any app? Today your Pico becomes a tiny web server!

Every time you open a website, your browser sends a **request** and the server sends back a **response** -- an HTML page. This conversation uses a language called **HTTP** (HyperText Transfer Protocol).

*Your instructor will explain how HTTP works, what a socket is, and how your Pico can be both the waiter and the kitchen in a restaurant analogy.*

---
## Components

Same wiring as Module 2! If your LED is still on the breadboard, you're ready to go.

| Component | Quantity | What It Does |
|-----------|----------|--------------|
| Pico 2W | 1 | Your web server + LED controller |
| LED | 1 | The light you'll control from your phone |
| 220-ohm resistor | 1 | Protects the LED from too much current |
| Jumper wires | 2 | Connections |
| Breadboard | 1 | Base for your circuit |

### Wiring Diagram

```
    Pico 2W
    +--------+
    |        |
    |   GP15 |----[ 220 ohm ]----[LED >|]---- GND
    |        |
    |    GND |---(breadboard ground rail)
    |        |
    +--------+
```

- **GP15** -> 220-ohm resistor -> LED long leg (anode)
- LED short leg (cathode) -> GND

---
## Code Along Step 1: Connect to WiFi (Starter Code)

We'll reuse the WiFi connection code from Module 15. Fill in your network name and password:

In [ ]:
import network
import time
from machine import Pin

# --- WiFi credentials ---
SSID = "_____"                            # Your instructor will provide this
PASSWORD = "_____"                         # Your instructor will provide this

# --- Set up LED ---
led = Pin(15, Pin.OUT)
led.value(0)                               # Start with LED off

# --- Connect to WiFi ---
wlan = network.WLAN(network.STA_IF)
wlan.active(True)
wlan.connect(SSID, PASSWORD)

print("Connecting to WiFi...")
timeout = 10
while not wlan.isconnected() and timeout > 0:
    print("  Waiting...", timeout)
    time.sleep(1)
    timeout -= 1

if wlan.isconnected():
    ip = wlan.ifconfig()[0]
    print("Connected! IP address:", ip)
    print("Open this in your browser: http://" + ip)
else:
    print("Failed to connect. Check SSID and password.")

Write down your Pico's IP address: ____________

You'll need it soon to open the web page!

---
## Code Along Step 2: Create a Socket (the Phone Line)

A **socket** is like opening a phone line. You create it, bind it to an address, and start listening for calls.

Fill in the blanks:

In [ ]:
import socket

# Get the address to listen on
# '0.0.0.0' means "accept connections from anyone"
# 80 is the standard port for web traffic (HTTP)
addr = socket.getaddrinfo('0.0.0.0', 80)[0][-1]

# Create the socket
s = socket.socket()

# Allow reuse of the address (helpful when restarting the code)
s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

# Bind the socket to our address
s.bind(_____)              # What variable holds our address?

# Start listening for connections
s.listen(_____)            # How many connections to queue? (just 1 is fine)

print("Web server is listening on", ip, "port 80")

Think of what just happened:
1. We created a socket (opened a phone line)
2. We bound it to an address (assigned it a phone number)
3. We started listening (picked up the phone and waited)

Now any browser on the same WiFi can "call" our Pico!

---
## Code Along Step 3: Build the Web Page

When a browser connects, we need to send back an HTML page. HTML is the language that web pages are written in.

Our page will have:
- A title
- The current LED status
- An ON button and an OFF button

Fill in the blanks for the button labels:

In [ ]:
def build_page(led_state):
    """Build an HTML page with ON/OFF buttons and LED status."""
    if led_state:
        status = "ON"
        status_color = "#4CAF50"       # Green
    else:
        status = "OFF"
        status_color = "#F44336"       # Red
    
    html = """<!DOCTYPE html>
<html>
<head>
    <title>Pico LED Dashboard</title>
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <style>
        body {
            font-family: Arial, sans-serif;
            text-align: center;
            background-color: #1a1a2e;
            color: white;
            margin: 0;
            padding: 20px;
        }
        h1 { font-size: 28px; margin-top: 30px; }
        .status {
            font-size: 24px;
            margin: 20px;
            padding: 15px;
            border-radius: 10px;
            background-color: """ + status_color + """;
        }
        .btn {
            display: inline-block;
            padding: 20px 40px;
            margin: 10px;
            font-size: 22px;
            border: none;
            border-radius: 10px;
            color: white;
            cursor: pointer;
            text-decoration: none;
        }
        .btn-on  { background-color: #4CAF50; }
        .btn-off { background-color: #F44336; }
    </style>
</head>
<body>
    <h1>Pico LED Dashboard</h1>
    <div class="status">LED is """ + status + """</div>
    <a href="/on"  class="btn btn-on">_____</a>
    <a href="/off" class="btn btn-off">_____</a>
    <p style="margin-top:30px; color:#888;">Circuit Explorers - Module 16</p>
</body>
</html>
"""
    return html

# Test it! Print the page to see the HTML
print(build_page(False))

Look at the HTML output. Can you spot:
- The title? ____________
- The status text? ____________
- The two links (`/on` and `/off`)? These are what the buttons will "visit" when clicked.

When the browser visits `/on`, we'll turn the LED on. When it visits `/off`, we'll turn it off. Simple!

---
## Code Along Step 4: The Main Server Loop

Now we put it all together. The server will:
1. Wait for a browser to connect
2. Read what the browser is asking for (the URL)
3. If the URL contains `/on`, turn the LED on
4. If the URL contains `/off`, turn the LED off
5. Send back the web page
6. Go back to step 1

Fill in the blanks:

In [ ]:
# --- Full Web Server ---
# (This cell combines everything -- run this as your final program)

import network
import socket
import time
from machine import Pin

# --- WiFi credentials ---
SSID = "_____"
PASSWORD = "_____"

# --- Set up LED ---
led = Pin(_____, Pin.OUT)                  # What GPIO pin is the LED on?
led.value(0)
led_state = False

# --- Connect to WiFi ---
wlan = network.WLAN(network.STA_IF)
wlan.active(True)
wlan.connect(SSID, PASSWORD)

print("Connecting to WiFi...")
while not wlan.isconnected():
    time.sleep(1)

ip = wlan.ifconfig()[0]
print("Connected! IP:", ip)

# --- Build page function ---
def build_page(led_state):
    if led_state:
        status = "ON"
        status_color = "#4CAF50"
    else:
        status = "OFF"
        status_color = "#F44336"
    
    html = """<!DOCTYPE html>
<html>
<head>
    <title>Pico LED Dashboard</title>
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <style>
        body { font-family: Arial, sans-serif; text-align: center;
               background-color: #1a1a2e; color: white; padding: 20px; }
        h1 { font-size: 28px; margin-top: 30px; }
        .status { font-size: 24px; margin: 20px; padding: 15px;
                  border-radius: 10px; background-color: """ + status_color + """; }
        .btn { display: inline-block; padding: 20px 40px; margin: 10px;
               font-size: 22px; border: none; border-radius: 10px;
               color: white; cursor: pointer; text-decoration: none; }
        .btn-on  { background-color: #4CAF50; }
        .btn-off { background-color: #F44336; }
    </style>
</head>
<body>
    <h1>Pico LED Dashboard</h1>
    <div class="status">LED is """ + status + """</div>
    <a href="/on"  class="btn btn-on">Turn ON</a>
    <a href="/off" class="btn btn-off">Turn OFF</a>
    <p style="margin-top:30px; color:#888;">Circuit Explorers - Module 16</p>
</body>
</html>
"""
    return html

# --- Create socket ---
addr = socket.getaddrinfo('0.0.0.0', 80)[0][-1]
s = socket.socket()
s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
s.bind(addr)
s.listen(1)

print("\n** Web server running! **")
print("Open your browser to: http://" + ip)
print("Press Ctrl+C to stop\n")

# --- Main server loop ---
try:
    while True:
        # Wait for a connection
        cl, client_addr = s.accept()
        print("Connection from", client_addr)
        
        # Read the request
        request = cl.recv(1024).decode()
        
        # Check what URL was requested
        if "/on" in request:
            led.value(_____)               # What value turns the LED on?
            led_state = True
            print("  LED turned ON")
        elif "/off" in request:
            led.value(_____)               # What value turns the LED off?
            led_state = _____              # What is the state now?
            print("  LED turned OFF")
        
        # Send the web page back
        response = build_page(led_state)
        cl.send("HTTP/1.1 200 OK\r\n")
        cl.send("Content-Type: text/html\r\n")
        cl.send("Connection: close\r\n\r\n")
        cl.send(response)
        cl.close()

except KeyboardInterrupt:
    print("\nServer stopped.")
    s.close()
    led.value(0)

### Try it!

1. Run the code above
2. Wait for "Web server running!" and the IP address
3. On your phone or laptop, open a browser
4. Type the IP address into the address bar (e.g., `http://192.168.1.42`)
5. You should see the dashboard with ON and OFF buttons!
6. Press the buttons and watch your LED!

Questions:
- Does the LED on your breadboard match what the web page says? ____________
- What do you see in the Pico's console when you press a button? ____________
- What is the URL in your browser's address bar after pressing ON? ____________

---
## Experiments

Try these modifications:

1. **Phone vs. computer:** Open the dashboard on your phone AND your laptop at the same time. Can both control the LED? Does the status update on one when you press a button on the other?

2. **Friend's device:** Ask a friend (on the same WiFi) to open your Pico's IP address in their browser. Can they control YOUR LED? How does that feel?

3. **Custom message:** Change the `<h1>` title text to something fun, like your name or a project name. Reload the page to see the change.

4. **Custom colors:** Try changing the background color from `#1a1a2e` to another color. Some ideas: `#0d1117` (dark), `#1b2838` (blue-dark), `#2d2d2d` (gray).

---
## Challenge: Multi-LED Dashboard

Add buttons for **3 LEDs** on different GPIO pins, or add a **status indicator** that shows whether each LED is on or off.

### Ideas:

- Wire up 3 LEDs on GP15, GP14, and GP13 (each with a 220-ohm resistor)
- Create separate ON/OFF buttons for each LED (e.g., `/led1on`, `/led1off`, `/led2on`, etc.)
- Show each LED's status on the page with a colored circle or emoji
- Add a "Toggle" button that switches an LED from on to off or off to on
- Add an "All ON" and "All OFF" button

### Hints:

```python
# Set up multiple LEDs
led1 = Pin(15, Pin.OUT)
led2 = Pin(14, Pin.OUT)
led3 = Pin(13, Pin.OUT)

# Check the URL for different commands
if "/led1on" in request:
    led1.value(1)
elif "/led1off" in request:
    led1.value(0)
# ... and so on for led2, led3
```

Ask your instructor for help if you get stuck!

---
## Vocabulary Recap

| Word | What It Means |
|------|---------------|
| **HTTP** | HyperText Transfer Protocol -- the language browsers and servers use to talk to each other |
| **Web server** | A program that listens for browser connections and sends back web pages |
| **HTML** | HyperText Markup Language -- the language web pages are written in |
| **Socket** | A connection endpoint -- like a phone line that lets two programs talk over a network |
| **Request** | What the browser sends to the server ("I want this page") |
| **Response** | What the server sends back to the browser (the HTML page) |
| **Port 80** | The standard "door number" for web traffic (HTTP) |
| **URL** | Uniform Resource Locator -- the address you type in a browser (like `http://192.168.1.42/on`) |

---
*Circuit Explorers -- Module 16: Web Dashboard*